# Subtask 1 — Basic Seq2Seq Translation

**Objective:** Implement a basic LSTM encoder–decoder for English → French translation, train on the supplied parallel corpus, evaluate on a held-out test split using corpus BLEU, and analyze visible failure modes.

This notebook intentionally **does not use attention**. Attention is introduced in Subtask 2 so that the experiments form a controlled progression.

### Pipeline
`Data → cleaning/tokenization → train-only vocab → Dataset/DataLoader → LSTM Encoder → LSTM Decoder → teacher-forced training → validation checkpoint → greedy decoding → BLEU + diagnostics`

In [ ]:
# Imports and reproducibility
import os, re, random, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Configuration

Adjust only the paths if the files are stored somewhere else. The validation split is used for checkpoint selection; the test split remains untouched until final evaluation.

In [ ]:
DATA_DIR = Path("/mnt/data")
TRAIN_ZIP = DATA_DIR / "en_fr_train(1).zip"
VAL_PATH = DATA_DIR / "en_fr_val(1).csv"
TEST_PATH = DATA_DIR / "en_fr_test(1).csv"

EXTRACT_DIR = DATA_DIR / "en_fr_train_extracted"
ARTIFACT_DIR = DATA_DIR / "exp1_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 1
DROPOUT = 0.20
BATCH_SIZE = 64
LR = 1e-3
N_EPOCHS = 20
CLIP = 1.0
TEACHER_FORCING_RATIO = 0.5
MIN_FREQ = 2
MAX_SRC_LEN = 80
MAX_TRG_LEN = 80
PATIENCE = 4

In [ ]:
# Extract training archive
EXTRACT_DIR.mkdir(exist_ok=True)

if TRAIN_ZIP.exists():
    with zipfile.ZipFile(TRAIN_ZIP, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Extracted:", TRAIN_ZIP)

train_csvs = sorted(EXTRACT_DIR.rglob("*.csv"))
print("CSV files found:", [str(p) for p in train_csvs[:10]])

if VAL_PATH.exists() and TEST_PATH.exists():
    print("Validation:", VAL_PATH)
    print("Test:", TEST_PATH)
else:
    raise FileNotFoundError("Check VAL_PATH and TEST_PATH.")

## 2. Load and inspect the parallel corpus

The notebook accepts training CSVs containing `en` and `fr`. If the ZIP contains more than one CSV, the first file containing both required columns is selected.

In [ ]:
def find_parallel_csv(paths):
    for p in paths:
        try:
            tmp = pd.read_csv(p)
            if {"en", "fr"}.issubset(tmp.columns):
                return p
        except Exception:
            pass
    raise FileNotFoundError("No CSV containing both 'en' and 'fr' was found.")

TRAIN_PATH = find_parallel_csv(train_csvs)
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

for df in (train_df, val_df, test_df):
    df.dropna(subset=["en", "fr"], inplace=True)
    df["en"] = df["en"].astype(str)
    df["fr"] = df["fr"].astype(str)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
display(train_df.head())

## 3. Cleaning and tokenization

The preprocessing is deliberately lightweight and identical across all splits. Vocabulary construction happens **only on the training data**.

In [ ]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text):
    text = normalize_text(text)
    return TOKEN_RE.findall(text)

train_df["en_tok"] = train_df["en"].map(tokenize)
train_df["fr_tok"] = train_df["fr"].map(tokenize)
val_df["en_tok"] = val_df["en"].map(tokenize)
val_df["fr_tok"] = val_df["fr"].map(tokenize)
test_df["en_tok"] = test_df["en"].map(tokenize)
test_df["fr_tok"] = test_df["fr"].map(tokenize)

print(train_df[["en", "fr", "en_tok", "fr_tok"]].head(3))

In [ ]:
# Basic length statistics
src_lengths = train_df["en_tok"].map(len)
trg_lengths = train_df["fr_tok"].map(len)

print("Source length median/max:", src_lengths.median(), src_lengths.max())
print("Target length median/max:", trg_lengths.median(), trg_lengths.max())

## 4. Vocabulary

The vocabulary contains four special tokens:

- `<pad>`: batch padding
- `<unk>`: unseen/rare words
- `<sos>`: decoder start
- `<eos>`: decoder stop

Rare tokens below `MIN_FREQ` are mapped to `<unk>`.

In [ ]:
PAD = "<pad>"
UNK = "<unk>"
SOS = "<sos>"
EOS = "<eos>"

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        freq = {}
        for toks in token_lists:
            for tok in toks:
                freq[tok] = freq.get(tok, 0) + 1

        self.itos = [PAD, UNK, SOS, EOS]
        self.itos += [tok for tok, c in sorted(freq.items()) if c >= min_freq]
        self.stoi = {tok:i for i, tok in enumerate(self.itos)}

        self.pad_idx = self.stoi[PAD]
        self.unk_idx = self.stoi[UNK]
        self.sos_idx = self.stoi[SOS]
        self.eos_idx = self.stoi[EOS]

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens, add_sos=False, add_eos=True):
        ids = []
        if add_sos:
            ids.append(self.sos_idx)
        ids.extend(self.stoi.get(t, self.unk_idx) for t in tokens)
        if add_eos:
            ids.append(self.eos_idx)
        return ids

    def decode(self, ids, stop_at_eos=True):
        out=[]
        for idx in ids:
            tok = self.itos[int(idx)]
            if tok == EOS and stop_at_eos:
                break
            if tok not in {PAD, SOS}:
                out.append(tok)
        return out

src_vocab = Vocab(train_df["en_tok"], min_freq=MIN_FREQ)
trg_vocab = Vocab(train_df["fr_tok"], min_freq=MIN_FREQ)

print("Source vocab:", len(src_vocab))
print("Target vocab:", len(trg_vocab))

## 5. Dataset and DataLoaders

Each source sentence is encoded as `tokens + <eos>`. The decoder receives `<sos> + target tokens`, while the loss target is `target tokens + <eos>`.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, df, src_vocab, trg_vocab, max_src_len=80, max_trg_len=80):
        self.df = df.reset_index(drop=True)
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_src_len = max_src_len
        self.max_trg_len = max_trg_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src = row["en_tok"][: self.max_src_len]
        trg = row["fr_tok"][: self.max_trg_len]

        src_ids = self.src_vocab.encode(src, add_sos=False, add_eos=True)
        trg_ids = self.trg_vocab.encode(trg, add_sos=True, add_eos=True)

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(trg_ids, dtype=torch.long)

def collate_fn(batch):
    srcs, trgs = zip(*batch)
    src_len = max(len(x) for x in srcs)
    trg_len = max(len(x) for x in trgs)

    src_pad = torch.full((len(batch), src_len), src_vocab.pad_idx, dtype=torch.long)
    trg_pad = torch.full((len(batch), trg_len), trg_vocab.pad_idx, dtype=torch.long)

    for i, (src, trg) in enumerate(zip(srcs, trgs)):
        src_pad[i, :len(src)] = src
        trg_pad[i, :len(trg)] = trg

    return src_pad, trg_pad

train_ds = TranslationDataset(train_df, src_vocab, trg_vocab, MAX_SRC_LEN, MAX_TRG_LEN)
val_ds = TranslationDataset(val_df, src_vocab, trg_vocab, MAX_SRC_LEN, MAX_TRG_LEN)
test_ds = TranslationDataset(test_df, src_vocab, trg_vocab, MAX_SRC_LEN, MAX_TRG_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

src_batch, trg_batch = next(iter(train_loader))
print(src_batch.shape, trg_batch.shape)

## 6. LSTM Encoder

The encoder processes the complete source sequence. Its final hidden and cell states initialize the decoder. This is the fixed-context architecture that Subtask 2 will improve with attention.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=src_vocab.pad_idx)
        self.rnn = nn.LSTM(
            emb_dim, hid_dim, n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

## 7. LSTM Decoder

The decoder generates one target token at a time. During training, teacher forcing optionally supplies the reference token from the previous position.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers=1, dropout=0.2):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=trg_vocab.pad_idx)
        self.rnn = nn.LSTM(
            emb_dim, hid_dim, n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

## 8. Seq2Seq Wrapper

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        output_dim = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, output_dim, device=self.device)
        _, hidden, cell = self.encoder(src)

        input_token = trg[:, 0]  # <sos>

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = output

            best_guess = output.argmax(1)
            input_token = trg[:, t] if random.random() < teacher_forcing_ratio else best_guess

        return outputs

## 9. Initialize model and optimization

In [ ]:
enc = Encoder(len(src_vocab), EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(len(trg_vocab), EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)

model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.pad_idx)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")

## 10. Training and validation

The best validation-loss checkpoint is retained. Gradient clipping is applied before each optimizer step.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, trg, TEACHER_FORCING_RATIO)

        output_dim = output.size(-1)
        loss = criterion(
            output[:, 1:].reshape(-1, output_dim),
            trg[:, 1:].reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    for src, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        output = model(src, trg, 0.0)

        output_dim = output.size(-1)
        loss = criterion(
            output[:, 1:].reshape(-1, output_dim),
            trg[:, 1:].reshape(-1)
        )
        total_loss += loss.item()

    return total_loss / len(loader)

best_val = float("inf")
patience_counter = 0
train_losses, val_losses = [], []

best_path = ARTIFACT_DIR / "basic_seq2seq_best.pt"

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load(best_path, map_location=DEVICE))

## 11. Loss curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Subtask 1 Training Curves")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 12. Greedy autoregressive translation

In [ ]:
@torch.no_grad()
def translate_tokens(model, src_tokens, max_len=MAX_TRG_LEN):
    model.eval()

    src_ids = src_vocab.encode(src_tokens[:MAX_SRC_LEN], add_sos=False, add_eos=True)
    src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    _, hidden, cell = model.encoder(src_tensor)
    input_token = torch.tensor([trg_vocab.sos_idx], dtype=torch.long, device=DEVICE)

    generated = []

    for _ in range(max_len):
        output, hidden, cell = model.decoder(input_token, hidden, cell)
        next_token = output.argmax(1).item()

        if next_token == trg_vocab.eos_idx:
            break

        generated.append(next_token)
        input_token = torch.tensor([next_token], dtype=torch.long, device=DEVICE)

    return trg_vocab.decode(generated)

def translate_text(model, text):
    return translate_tokens(model, tokenize(text))

examples = test_df.sample(min(10, len(test_df)), random_state=SEED)
for _, row in examples.iterrows():
    pred = translate_text(model, row["en"])
    print("SRC :", row["en"])
    print("REF :", row["fr"])
    print("PRED:", " ".join(pred))
    print("-"*90)

## 13. Corpus BLEU on the held-out test set

BLEU is computed only after the final checkpoint has been selected using validation loss. This is the number that should be carried into Subtask 2 as the no-attention baseline.

In [ ]:
references = []
hypotheses = []

for _, row in test_df.iterrows():
    ref = tokenize(row["fr"])
    hyp = translate_text(model, row["en"])

    references.append([ref])
    hypotheses.append(hyp)

smooth = SmoothingFunction().method1
corpus_bleu_score = corpus_bleu(references, hypotheses)

print(f"Subtask 1 corpus BLEU: {corpus_bleu_score:.6f}")

# Save the actual result for Subtask 2
with open(ARTIFACT_DIR / "subtask1_bleu.txt", "w", encoding="utf-8") as f:
    f.write(str(corpus_bleu_score))

## 14. Sentence-level diagnostics

The following diagnostics are intentionally retained for the assignment report:
- sentence BLEU for ranking examples,
- BLEU by source length,
- unknown-token prediction rate,
- repetition rate,
- prediction/reference length ratio,
- best and worst examples.

In [ ]:
def repetition_rate(tokens):
    if len(tokens) < 2:
        return 0.0
    repeated = sum(a == b for a, b in zip(tokens, tokens[1:]))
    return repeated / (len(tokens) - 1)

rows = []
for _, row in test_df.iterrows():
    ref = tokenize(row["fr"])
    pred = translate_text(model, row["en"])
    rows.append({
        "source": row["en"],
        "reference": row["fr"],
        "prediction": " ".join(pred),
        "source_len": len(tokenize(row["en"])),
        "ref_len": len(ref),
        "pred_len": len(pred),
        "sentence_bleu": sentence_bleu([ref], pred, smoothing_function=smooth),
        "unk_rate": pred.count(UNK) / max(1, len(pred)),
        "repetition_rate": repetition_rate(pred),
        "length_ratio": len(pred) / max(1, len(ref))
    })

results = pd.DataFrame(rows)
display(results.head())

In [ ]:
# BLEU by source length bucket
bins = [-1, 10, 20, 30, 40, 10**9]
labels = ["0-10", "11-20", "21-30", "31-40", "41+"]
results["length_bucket"] = pd.cut(results["source_len"], bins=bins, labels=labels)

length_summary = (
    results.groupby("length_bucket", observed=True)["sentence_bleu"]
    .agg(["mean", "count"])
    .reset_index()
)
display(length_summary)

plt.figure(figsize=(8,4))
plt.bar(length_summary["length_bucket"].astype(str), length_summary["mean"])
plt.xlabel("Source sentence length")
plt.ylabel("Mean sentence BLEU")
plt.title("BLEU vs Source Length")
plt.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
print("Mean unknown-token rate:", results["unk_rate"].mean())
print("Mean repetition rate:", results["repetition_rate"].mean())
print("Mean prediction/reference length ratio:", results["length_ratio"].mean())

print("\nBest examples:")
display(results.sort_values("sentence_bleu", ascending=False)[
    ["source","reference","prediction","sentence_bleu"]
].head(5))

print("\nWorst examples:")
display(results.sort_values("sentence_bleu")[
    ["source","reference","prediction","sentence_bleu"]
].head(5))

## 15. Qualitative failure-mode analysis

When interpreting failures, look specifically for:

1. **Long sentences:** information may be lost because the encoder compresses the full source into a fixed state.
2. **Rare words:** unseen or low-frequency words may become `<unk>` or be translated incorrectly.
3. **Repetition:** recurrent generation can loop on common tokens.
4. **Length mismatch:** the decoder may terminate too early or produce excessively long outputs.
5. **Semantic drift:** a locally plausible sequence may not preserve the complete source meaning.

These observations motivate Subtask 2, where attention gives the decoder direct access to encoder states.

## 16. Save predictions and final artifacts

In [ ]:
results.to_csv(ARTIFACT_DIR / "subtask1_test_predictions.csv", index=False)

summary = {
    "corpus_bleu": float(corpus_bleu_score),
    "best_val_loss": float(best_val),
    "embedding_dim": EMB_DIM,
    "hidden_dim": HID_DIM,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "teacher_forcing_ratio": TEACHER_FORCING_RATIO,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
}

with open(ARTIFACT_DIR / "subtask1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved artifacts to:", ARTIFACT_DIR)
print(summary)

## 17. Final interpretation

This experiment is the controlled no-attention baseline. Its purpose is not to maximize translation quality through every available technique. Its purpose is to establish a reproducible fixed-context Seq2Seq system and document the limitations that attention is expected to address.

**Next experiment:** Subtask 2 adds Bahdanau attention and compares decoding strategies while keeping the translation problem and evaluation protocol aligned.